# `nanoddpm-pro` – From‑Scratch Diffusion Models for MNIST & CIFAR‑10

## Overview
This notebook implements a **unified Denoising Diffusion Probabilistic Model (DDPM)** from scratch, supporting both **MNIST** and **CIFAR‑10** datasets. It is designed as an educational yet powerful toolkit for understanding modern diffusion‑based generative models.

## Key Features
- **Flexible dataset support** – switch between MNIST and CIFAR‑10 with a single variable.
- **Two sampling frameworks**:
  - **DDIM** – deterministic, fast sampling (10–50 steps)
  - **EDM** – continuous‑time ODE sampling with **Euler / Heun** solvers
- **Two training targets** – `epsilon` (noise prediction) or `v`‑prediction (used in Stable Diffusion 2+)
- **Classifier‑Free Guidance (CFG)** – control class conditioning strength at inference.
- **Exponential Moving Average (EMA)** – improves sample quality.
- **Lightweight evaluation metrics** – PCA‑FID, Sobel sharpness, intensity KL divergence (no InceptionV3 needed).
- **Modular architecture** – Mini‑UNet with sinusoidal time/class embeddings, resblocks, skip connections.

## How to Use
1. Set your parameters in **Cell 2** (dataset, sampler, target, CFG scale, etc.).
2. Run all cells sequentially.
3. Training will start automatically, and after each epoch you'll see loss, PCA‑FID, and other metrics.
4. Final visualisations include:
   - Training loss & PCA‑FID curves
   - Generated samples (2 per class) decoded with **both** epsilon and v formulas – so you can see how the same trained model behaves with different decoders.
   - (For EDM) a denoising trajectory plot showing the reverse process from pure noise to clean image.

## Requirements
- `torch`, `torchvision`, `matplotlib`, `numpy`, `tqdm`
- A GPU is recommended (free Colab GPU works fine). CPU will be very slow.


## References
- DDPM: Ho et al. (2020)
- DDIM: Song et al. (2021)
- EDM: Karras et al. (2022)
- v‑prediction: Salimans & Ho (2022)
- PCA‑FID: educational adaptation of the standard FID metric

**Enjoy exploring diffusion models from the ground up!**

# 1. Imports and Configuration

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as T
import matplotlib.pyplot as plt
import numpy as np
import math
import json
import copy
from tqdm import trange
import torch.nn.functional as F

# Set seed and device
torch.manual_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

In [ ]:
# ========== CONFIGURATION ==========
DATASET = 'mnist'          # 'mnist' or 'cifar10'
EPOCHS = 20
BATCH_SIZE = 32
RESIZE = 32                  # only used for CIFAR‑10 (MNIST stays 28)
CFG_SCALE = 4.0

# Sampling framework
SAMPLER = 'edm'              # 'ddim' or 'edm'
TARGET = 'v'           # 'epsilon' or 'v'
USE_EMA = True
EMA_DECAY = 0.999

# DDIM specific
T_STEPS = 1000
DDIM_SAMPLE_STEPS = 50
BETA_SCHEDULE = 'cosine'     # 'linear' or 'cosine'

# EDM specific
P_MEAN, P_STD = -1.2, 1.2
SIGMA_MIN, SIGMA_MAX = 0.002, 80.0
EDM_SAMPLE_STEPS = 20
SOLVER = 'euler'             # 'euler' or 'heun'

print(f"▶ Dataset: {DATASET} | Sampler: {SAMPLER} | Target: {TARGET} | CFG: {CFG_SCALE}")

# 2. Dataset Loading & Preparation

In [ ]:
if DATASET == 'mnist':
    img_channels = 1
    img_size = 28
    num_classes = 10
    transform = T.Compose([T.ToTensor(), T.Normalize([0.5], [0.5])])
    dataset = torchvision.datasets.MNIST(
        root="./data", train=True, download=True, transform=transform
    )
else:  # cifar10
    img_channels = 3
    img_size = RESIZE
    num_classes = 10
    transform = T.Compose([
        T.Resize(RESIZE),
        T.ToTensor(),
        T.Normalize([0.5]*3, [0.5]*3)
    ])
    dataset = torchvision.datasets.CIFAR10(
        root="./data", train=True, download=True, transform=transform
    )

loader = torch.utils.data.DataLoader(
    dataset, batch_size=BATCH_SIZE, shuffle=True,
    num_workers=2, pin_memory=True
)

# Fixed reference batch for evaluation
real_batch, real_labels = next(
    iter(torch.utils.data.DataLoader(dataset, batch_size=256, shuffle=False))
)
real_batch, real_labels = real_batch.to(device), real_labels.to(device)
print(f"Dataset loaded. Sample shape: {dataset[0][0].shape}")

# 3. Diffusion Model (Mini‑UNet + Time/Class Embedding)

In [ ]:
def sinusoidal_embedding(t, dim, max_period=10000):
    half = dim // 2
    freqs = torch.exp(-math.log(max_period) * torch.arange(half, dtype=torch.float32, device=t.device) / half)
    args = t[:, None].float() * freqs[None, :]
    return torch.cat([torch.cos(args), torch.sin(args)], dim=1)

class ResBlock(nn.Module):
    def __init__(self, in_ch, out_ch, time_dim, num_classes=11, dropout=0.1):
        super().__init__()
        self.num_groups_gn = min(8, out_ch)
        self.conv1 = nn.Conv2d(in_ch, out_ch, 3, padding=1)
        self.norm1 = nn.GroupNorm(self.num_groups_gn, out_ch)
        self.conv2 = nn.Conv2d(out_ch, out_ch, 3, padding=1)
        self.norm2 = nn.GroupNorm(self.num_groups_gn, out_ch)
        self.time_mlp = nn.Sequential(nn.SiLU(), nn.Linear(time_dim, out_ch))
        self.class_emb = nn.Embedding(num_classes, time_dim)
        self.class_proj = nn.Linear(time_dim, out_ch)
        self.dropout = nn.Dropout(dropout)
        self.skip = nn.Conv2d(in_ch, out_ch, 1) if in_ch != out_ch else nn.Identity()
        self.time_dim = time_dim

    def forward(self, x, t, labels=None):
        h = F.silu(self.norm1(self.conv1(x)))
        t_emb = self.time_mlp(sinusoidal_embedding(t, self.time_dim))
        if labels is None:
            c_emb = torch.zeros_like(t_emb)
        else:
            c_emb = self.class_proj(self.dropout(self.class_emb(labels)))
        h = h + (t_emb + c_emb)[:, :, None, None]
        return F.silu(self.norm2(self.conv2(h))) + self.skip(x)

class MiniUNet(nn.Module):
    def __init__(self, in_ch=3, out_ch=3, time_dim=128, ch=32):
        super().__init__()
        self.down1 = ResBlock(in_ch, ch, time_dim)
        self.down2 = ResBlock(ch, ch*2, time_dim)
        self.mid = ResBlock(ch*2, ch*2, time_dim)
        self.up1 = ResBlock(ch*4, ch, time_dim)
        self.up2 = ResBlock(ch*2, out_ch, time_dim)
        self.out = nn.Conv2d(out_ch, out_ch, 1)

    def forward(self, x, t, labels=None):
        d1 = self.down1(x, t, labels)
        d2 = self.down2(F.avg_pool2d(d1, 2), t, labels)
        x = self.mid(F.avg_pool2d(d2, 2), t, labels)
        x = self.up1(torch.cat([F.interpolate(x, scale_factor=2), d2], 1), t, labels)
        x = self.up2(torch.cat([F.interpolate(x, scale_factor=2), d1], 1), t, labels)
        return self.out(x)

# 4. Helper Functions (Metrics, Schedulers, EMA, Wrappers)

In [ ]:
# ================ Metrics (PCA-FID, Sobel, KL) ============================
def pca_fid(real, gen, n_components=32):
    r = real.view(real.shape[0], -1).cpu().double()
    g = gen.view(gen.shape[0], -1).cpu().double()
    data = torch.cat([r, g], 0)
    mean = data.mean(0, keepdim=True)
    U, S, V = torch.pca_lowrank(data - mean, q=n_components)
    proj = (data - mean) @ V
    r_p, g_p = proj[:real.shape[0]], proj[real.shape[0]:]
    mu_r, mu_g = r_p.mean(0), g_p.mean(0)
    var_r, var_g = r_p.var(0)+1e-6, g_p.var(0)+1e-6
    return ((mu_r-mu_g)**2).sum().item() + (var_r+var_g-2*torch.sqrt(var_r*var_g)).sum().item()

def sobel_grad(imgs):
    sx = torch.tensor([[-1,0,1],[-2,0,2],[-1,0,1]], dtype=torch.float32, device=imgs.device).view(1,1,3,3)
    sy = torch.tensor([[-1,-2,-1],[0,0,0],[1,2,1]], dtype=torch.float32, device=imgs.device).view(1,1,3,3)
    gx = F.conv2d(imgs, sx, padding=1)
    gy = F.conv2d(imgs, sy, padding=1)
    return torch.sqrt(gx**2 + gy**2 + 1e-8).mean().item()

def intensity_kl(real, gen, bins=50):
    r = real.cpu().view(-1).clamp(-1,1).numpy()
    g = gen.cpu().view(-1).clamp(-1,1).numpy()
    hr, _ = np.histogram(r, bins=bins, range=(-1,1), density=True)
    hg, _ = np.histogram(g, bins=bins, range=(-1,1), density=True)
    hr, hg = hr+1e-8, hg+1e-8
    hr /= hr.sum()
    hg /= hg.sum()
    return np.sum(hg * np.log(hg/hr)).item()

# =================== Noise schedules ====================================
if SAMPLER == 'ddim':
    if BETA_SCHEDULE == 'linear':
        beta = torch.linspace(1e-4, 0.02, T_STEPS, device=device)
    else:  # cosine
        def cosine_beta_schedule(T, s=0.008):
            x = torch.linspace(0, T, T+1, device=device)
            alphas_bar = torch.cos(((x / T) + s) / (1 + s) * math.pi * 0.5) ** 2
            alphas_bar = alphas_bar / alphas_bar[0]
            beta = 1 - (alphas_bar[1:] / alphas_bar[:-1])
            return torch.clamp(beta, 1e-5, 0.999)
        beta = cosine_beta_schedule(T_STEPS)
    alpha = 1.0 - beta
    alpha_bar = torch.cumprod(alpha, dim=0)
    sqrt_alpha_bar = torch.sqrt(alpha_bar)
    sqrt_one_minus_alpha_bar = torch.sqrt(1.0 - alpha_bar)
else:  # EDM – only helper for sigma sampling
    def sample_sigmas(n, dev, P_mean=P_MEAN, P_std=P_STD):
        return torch.exp(P_mean + P_std * torch.randn(n, device=dev))

# =============== EMA update ===========================================
def update_ema(model, ema_model, decay=0.999):
    with torch.no_grad():
        for p, ema_p in zip(model.parameters(), ema_model.parameters()):
            ema_p.mul_(decay).add_(p, alpha=1 - decay)

# ============ EDM preconditioning wrapper (target‑aware) ===============
class EDMWrapper(nn.Module):
    def __init__(self, model, target='epsilon'):
        super().__init__()
        self.model = model
        self.target = target
    def forward(self, x, sigma, labels=None):
        c_in = 1 / torch.sqrt(sigma**2 + 1)
        c_noise = torch.log(sigma) / 4
        model_out = self.model(x * c_in[:,None,None,None], c_noise, labels)
        c_skip = 1 / (sigma**2 + 1)
        c_out = sigma / torch.sqrt(sigma**2 + 1)
        if self.target == 'epsilon':
            return c_skip[:,None,None,None] * x + c_out[:,None,None,None] * model_out
        else:
            return model_out  # v-prediction

# 5. Samplers (DDIM / EDM)

In [ ]:
# =================== DDIM sampler (epsilon / v) ===============================
@torch.no_grad()
def sample_ddim(n, labels, cfg_scale=1.0, ddim_steps=50, target='epsilon'):
    model_eval = ema_model if USE_EMA else model
    model_eval.eval()
    x = torch.randn(n, img_channels, img_size, img_size, device=device)
    step_size = max(1, T_STEPS // ddim_steps)
    for t in reversed(range(0, T_STEPS, step_size)):
        t_t = torch.full((n,), t, device=device, dtype=torch.long)
        eps_u = model_eval(x, t_t, None)
        eps_c = model_eval(x, t_t, labels)
        eps_pred = eps_u + cfg_scale * (eps_c - eps_u)
        sqrt_ab = sqrt_alpha_bar[t]
        sqrt_1m = sqrt_one_minus_alpha_bar[t]
        # decode to x0
        if target == 'epsilon':
            pred_x0 = (x - sqrt_1m * eps_pred) / sqrt_ab
        else:
            pred_x0 = sqrt_ab * x - sqrt_1m * eps_pred
        pred_x0 = torch.clamp(pred_x0, -1.0, 1.0)
        a_prev = alpha[t - step_size] if t >= step_size else alpha[0]
        sqrt_ap = torch.sqrt(a_prev)
        sqrt_1map = torch.sqrt(1.0 - a_prev)
        x = sqrt_ap * pred_x0 + sqrt_1map * eps_pred
        x = torch.clamp(x, -1.0, 1.0)
    return x

# ================= EDM sampler (Euler / Heun, epsilon / v) ====================
@torch.no_grad()
def edm_sampler(wrapper, n, labels, cfg=1.0, steps=20, solver='euler', target='epsilon'):
    model_eval = wrapper if not USE_EMA else EDMWrapper(ema_model, target=target)
    model_eval.eval()
    sigmas = torch.linspace(SIGMA_MAX, SIGMA_MIN, steps, device=device)
    x = torch.randn(n, img_channels, img_size, img_size, device=device) * sigmas[0]
    for s, s_next in zip(sigmas[:-1], sigmas[1:]):
        s_vec = torch.full((n,), s, device=device)
        D_u = model_eval(x, s_vec, None)
        D_c = model_eval(x, s_vec, labels)
        D = D_u + cfg * (D_c - D_u)
        # decode to x0
        if target == 'epsilon':
            x0_pred = D
        else:
            c_skip = 1 / (s**2 + 1)
            c_out = s / torch.sqrt(s**2 + 1)
            x0_pred = c_skip * x - c_out * D
        if solver == 'euler':
            x = x + (s_next - s) * (x - x0_pred) / s
        else:  # heun
            x_pred = x + (s_next - s) * (x - x0_pred) / s
            if s_next > 1e-5:
                s_next_vec = torch.full((n,), s_next, device=device)
                D_u_p = model_eval(x_pred, s_next_vec, None)
                D_c_p = model_eval(x_pred, s_next_vec, labels)
                D_p = D_u_p + cfg * (D_c_p - D_u_p)
                if target == 'epsilon':
                    x0_pred_p = D_p
                else:
                    c_skip_p = 1 / (s_next**2 + 1)
                    c_out_p = s_next / torch.sqrt(s_next**2 + 1)
                    x0_pred_p = c_skip_p * x_pred - c_out_p * D_p
                x = x + (s_next - s) * ((x - x0_pred)/s + (x_pred - x0_pred_p)/s_next) / 2
            else:
                x = x_pred
        x = torch.clamp(x, -1.0, 1.0)
    return x

# 6. Training and Evaluation Functions

In [ ]:
# ============= Training function (unified for DDIM / EDM) =====================
def train_one_epoch(model, loader, optimizer, epoch, is_edm):
    model.train()
    total_loss = 0
    count = 0
    for imgs, labels in loader:
        imgs, labels = imgs.to(device), labels.to(device)
        # random unconditional masking (10%)
        train_labels = labels.clone()
        mask_uncond = torch.rand_like(labels.float()) < 0.1
        train_labels[mask_uncond] = num_classes   # unconditional index

        if not is_edm:   # DDIM branch
            t = torch.randint(0, T_STEPS, (imgs.shape[0],), device=device)
            sqrt_ab = sqrt_alpha_bar[t][:, None, None, None]
            sqrt_1m = sqrt_one_minus_alpha_bar[t][:, None, None, None]
            eps = torch.randn_like(imgs)
            xt = sqrt_ab * imgs + sqrt_1m * eps
            pred = model(xt, t, train_labels)

            if TARGET == 'epsilon':
                # SNR-weighted loss
                snr = alpha_bar[t] / (1 - alpha_bar[t] + 1e-8)
                weight = snr / (snr + 1)
                loss = F.mse_loss(pred, eps, reduction='none').mean(dim=(1,2,3))
                loss = (loss * weight).mean()
            else:  # v-prediction
                target_v = sqrt_ab * eps - sqrt_1m * imgs
                loss = F.mse_loss(pred, target_v)
        else:  # EDM branch
            sigma = sample_sigmas(imgs.shape[0], device)
            noise = torch.randn_like(imgs)
            x_sigma = imgs + sigma[:, None, None, None] * noise
            pred = edm_wrapper(x_sigma, sigma, train_labels)

            if TARGET == 'epsilon':
                c_out = sigma / torch.sqrt(sigma**2 + 1)
                loss_weight = (1.0 / (c_out**2 + 1e-8)).view(-1,1,1,1)
                loss = (loss_weight * F.mse_loss(pred, imgs, reduction='none')).mean()
            else:  # v-prediction
                c_skip = 1 / (sigma**2 + 1)
                c_out = sigma / torch.sqrt(sigma**2 + 1)
                eps = (x_sigma - imgs) / (sigma[:,None,None,None] + 1e-8)
                target_v = c_out[:,None,None,None] * eps - c_skip[:,None,None,None] * imgs
                loss = F.mse_loss(pred, target_v)

        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        if USE_EMA:
            update_ema(model, ema_model, EMA_DECAY)

        total_loss += loss.item() * imgs.shape[0]
        count += imgs.shape[0]

    return total_loss / count

# ================ Full training loop with evaluation ==========================
def run_training(epochs, is_edm):
    metrics_log = []
    for epoch in trange(1, epochs+1, desc="Training"):
        avg_loss = train_one_epoch(model, loader, optimizer, epoch, is_edm)

        # Evaluation every epoch
        model_eval = ema_model if USE_EMA else model
        model_eval.eval()
        with torch.no_grad():
            if not is_edm:
                gen = sample_ddim(128, real_labels[:128], cfg_scale=CFG_SCALE,
                                  ddim_steps=DDIM_SAMPLE_STEPS, target=TARGET)
            else:
                gen = edm_sampler(edm_wrapper, 128, real_labels[:128], cfg=CFG_SCALE,
                                  steps=EDM_SAMPLE_STEPS, solver=SOLVER, target=TARGET)
            fid = pca_fid(real_batch[:128], gen)
            grad = sobel_grad(gen)
            kl = intensity_kl(real_batch[:128], gen)

        metrics_log.append({
            'epoch': epoch,
            'loss': avg_loss,
            'pca_fid': fid,
            'sobel_grad': grad,
            'kl_div': kl
        })
        print(f"Epoch {epoch:02d} | Loss: {avg_loss:.4f} | PCA-FID: {fid:.2f} | Grad: {grad:.3f} | KL: {kl:.4f}")
    return metrics_log

# 7. Visualization Functions

In [ ]:
# =========: Single run plot (loss + PCA-FID) ==================================
def plot_single_run(metrics_log, title_suffix=""):
    epochs = [m['epoch'] for m in metrics_log]
    fig, axs = plt.subplots(1, 2, figsize=(12, 3))
    axs[0].plot(epochs, [m['loss'] for m in metrics_log], marker='o')
    axs[0].set_title('Training Loss')
    axs[0].grid(alpha=0.3)
    axs[1].plot(epochs, [m['pca_fid'] for m in metrics_log], marker='s', color='orange')
    axs[1].set_title(f'PCA-FID ({TARGET}, {SAMPLER.upper()}) ↓ better')
    axs[1].grid(alpha=0.3)
    plt.suptitle(title_suffix)
    plt.tight_layout()
    plt.show()

# ======= Final samples comparison (2 per class, both decoder formulas) ========
def plot_final_samples_comparison():
    model_eval = ema_model if USE_EMA else model
    model_eval.eval()
    with torch.no_grad():
        # Generate using either DDIM or EDM
        if SAMPLER == 'ddim':
            gen_eps = sample_ddim(20, torch.arange(10).repeat_interleave(2).to(device),
                                  cfg_scale=CFG_SCALE, ddim_steps=DDIM_SAMPLE_STEPS, target='epsilon')
            gen_v = sample_ddim(20, torch.arange(10).repeat_interleave(2).to(device),
                                cfg_scale=CFG_SCALE, ddim_steps=DDIM_SAMPLE_STEPS, target='v')
        else:
            gen_eps = edm_sampler(edm_wrapper, 20, torch.arange(10).repeat_interleave(2).to(device),
                                  cfg=CFG_SCALE, steps=EDM_SAMPLE_STEPS, solver=SOLVER, target='epsilon')
            gen_v = edm_sampler(edm_wrapper, 20, torch.arange(10).repeat_interleave(2).to(device),
                                cfg=CFG_SCALE, steps=EDM_SAMPLE_STEPS, solver=SOLVER, target='v')

        # Plot epsilon-decoded
        grid_eps = torchvision.utils.make_grid(gen_eps, nrow=10, normalize=True, value_range=(-1,1))
        plt.figure(figsize=(12, 2.5))
        plt.imshow(grid_eps.permute(1,2,0).cpu().numpy())
        plt.axis('off')
        plt.title(f'Samples decoded with epsilon (trained target: {TARGET})')
        plt.tight_layout()
        plt.show()

        # Plot v-decoded
        grid_v = torchvision.utils.make_grid(gen_v, nrow=10, normalize=True, value_range=(-1,1))
        plt.figure(figsize=(12, 2.5))
        plt.imshow(grid_v.permute(1,2,0).cpu().numpy())
        plt.axis('off')
        plt.title(f'Samples decoded with v (trained target: {TARGET})')
        plt.tight_layout()
        plt.show()

# 8. Execution Workflow


## EDM + noise (ɛ)

In [ ]:
# ============ Initialize model, optimizer, and EMA ============================
model = MiniUNet(in_ch=img_channels, out_ch=img_channels, time_dim=128, ch=32).to(device)
if SAMPLER == 'edm':
    edm_wrapper = EDMWrapper(model, target=TARGET).to(device)
optimizer = optim.Adam(model.parameters(), lr=2e-3)
ema_model = copy.deepcopy(model) if USE_EMA else None
print(f"▶ Model parameters: {sum(p.numel() for p in model.parameters()):,}")

# ================== Run training ==============================================
is_edm = (SAMPLER == 'edm')
print(f"▶ Dataset:{DATASET} | Sampler: {SAMPLER} | Target: {TARGET} | Beta Schedule: {BETA_SCHEDULE} | Solver: {SOLVER}")
metrics = run_training(epochs=EPOCHS, is_edm=is_edm)

# Save metrics
with open('nanoddpm_pro_metrics.json', 'w') as f:
    json.dump(metrics, f, indent=2)
print("Metrics saved to nanoddpm_pro_metrics.json")

# =============== Visualize results ============================================
plot_single_run(metrics, title_suffix=f"{DATASET} | {SAMPLER.upper()} | {TARGET}")
plot_final_samples_comparison()

# ===== Optional additional visualisation – trajectory (only for EDM) ==========
if SAMPLER == 'edm':
    @torch.no_grad()
    def sample_trajectory(n=9, sigmas_to_plot=[80, 40, 20, 10, 2, 0.002], cls_idx=5):
        model_eval = ema_model if USE_EMA else model
        model_eval.eval()
        labels = torch.full((n,), cls_idx, device=device)
        sigmas = torch.linspace(SIGMA_MAX, SIGMA_MIN, EDM_SAMPLE_STEPS, device=device)
        x = torch.randn(n, img_channels, img_size, img_size, device=device) * sigmas[0]
        trajectory_map = {s: None for s in sigmas_to_plot}
        for s, s_next in zip(sigmas[:-1], sigmas[1:]):
            s_vec = torch.full((n,), s, device=device)
            D_u = edm_wrapper(x, s_vec, None)
            D_c = edm_wrapper(x, s_vec, labels)
            D = D_u + CFG_SCALE * (D_c - D_u)
            if TARGET == 'epsilon':
                x0_pred = D
            else:
                c_skip = 1 / (s**2 + 1)
                c_out = s / torch.sqrt(s**2 + 1)
                x0_pred = c_skip * x - c_out * D
            x = x + (s_next - s) * (x - x0_pred) / s
            x = torch.clamp(x, -1.0, 1.0)
            for target_s in sigmas_to_plot:
                if abs(s.item() - target_s) < 2.0:
                    trajectory_map[target_s] = x.clone().cpu()
        valid_sigmas = sorted([s for s, v in trajectory_map.items() if v is not None], reverse=True)
        valid_imgs = [trajectory_map[s] for s in valid_sigmas]
        fig, axes = plt.subplots(1, len(valid_sigmas), figsize=(15,2))
        if len(valid_sigmas) == 1:
            axes = [axes]
        for i, (sigma_val, img) in enumerate(zip(valid_sigmas, valid_imgs)):
            grid = torchvision.utils.make_grid(img, nrow=3, normalize=True, value_range=(-1,1))
            axes[i].imshow(grid.permute(1,2,0).cpu().numpy())
            axes[i].set_title(f"σ={sigma_val:.1f}")
            axes[i].axis('off')
        plt.suptitle(f"Denoising trajectory – class {cls_idx} | {SOLVER} | {TARGET}")
        plt.tight_layout()
        plt.show()

    print("Trajectory example (class 5):")
    sample_trajectory(cls_idx=5)

print("✅ Done. All cells executed.")

## DDIM + noise (ɛ)

In [ ]:
SAMPLER = 'ddim'
TARGET = 'epsilon'
BETA_SCHEDULE = 'cosine'
SOLVER = 'euler'

DDIM_SAMPLE_STEPS = 50

In [ ]:
# =================== Noise schedules ====================================
if SAMPLER == 'ddim':
    if BETA_SCHEDULE == 'linear':
        beta = torch.linspace(1e-4, 0.02, T_STEPS, device=device)
    else:  # cosine
        def cosine_beta_schedule(T, s=0.008):
            x = torch.linspace(0, T, T+1, device=device)
            alphas_bar = torch.cos(((x / T) + s) / (1 + s) * math.pi * 0.5) ** 2
            alphas_bar = alphas_bar / alphas_bar[0]
            beta = 1 - (alphas_bar[1:] / alphas_bar[:-1])
            return torch.clamp(beta, 1e-5, 0.999)
        beta = cosine_beta_schedule(T_STEPS)
    alpha = 1.0 - beta
    alpha_bar = torch.cumprod(alpha, dim=0)
    sqrt_alpha_bar = torch.sqrt(alpha_bar)
    sqrt_one_minus_alpha_bar = torch.sqrt(1.0 - alpha_bar)
else:  # EDM – only helper for sigma sampling
    def sample_sigmas(n, dev, P_mean=P_MEAN, P_std=P_STD):
        return torch.exp(P_mean + P_std * torch.randn(n, device=dev))

In [ ]:
# ============ Initialize model, optimizer, and EMA ============================
model = MiniUNet(in_ch=img_channels, out_ch=img_channels, time_dim=128, ch=32).to(device)
if SAMPLER == 'edm':
    edm_wrapper = EDMWrapper(model, target=TARGET).to(device)
optimizer = optim.Adam(model.parameters(), lr=2e-3)
ema_model = copy.deepcopy(model) if USE_EMA else None
print(f"▶ Model parameters: {sum(p.numel() for p in model.parameters()):,}")

# ================== Run training ==============================================
is_edm = (SAMPLER == 'edm')
print(f"▶ Dataset:{DATASET} | Sampler: {SAMPLER} | Target: {TARGET} | Beta Schedule: {BETA_SCHEDULE} | Solver: {SOLVER}")
metrics = run_training(epochs=EPOCHS, is_edm=is_edm)

# Save metrics
with open('nanoddpm_pro_metrics.json', 'w') as f:
    json.dump(metrics, f, indent=2)
print("Metrics saved to nanoddpm_pro_metrics.json")

# =============== Visualize results ============================================
plot_single_run(metrics, title_suffix=f"{DATASET} | {SAMPLER.upper()} | {TARGET}")
plot_final_samples_comparison()

# ===== Optional additional visualisation – trajectory (only for EDM) ==========
if SAMPLER == 'edm':
    @torch.no_grad()
    def sample_trajectory(n=9, sigmas_to_plot=[80, 40, 20, 10, 2, 0.002], cls_idx=5):
        model_eval = ema_model if USE_EMA else model
        model_eval.eval()
        labels = torch.full((n,), cls_idx, device=device)
        sigmas = torch.linspace(SIGMA_MAX, SIGMA_MIN, EDM_SAMPLE_STEPS, device=device)
        x = torch.randn(n, img_channels, img_size, img_size, device=device) * sigmas[0]
        trajectory_map = {s: None for s in sigmas_to_plot}
        for s, s_next in zip(sigmas[:-1], sigmas[1:]):
            s_vec = torch.full((n,), s, device=device)
            D_u = edm_wrapper(x, s_vec, None)
            D_c = edm_wrapper(x, s_vec, labels)
            D = D_u + CFG_SCALE * (D_c - D_u)
            if TARGET == 'epsilon':
                x0_pred = D
            else:
                c_skip = 1 / (s**2 + 1)
                c_out = s / torch.sqrt(s**2 + 1)
                x0_pred = c_skip * x - c_out * D
            x = x + (s_next - s) * (x - x0_pred) / s
            x = torch.clamp(x, -1.0, 1.0)
            for target_s in sigmas_to_plot:
                if abs(s.item() - target_s) < 2.0:
                    trajectory_map[target_s] = x.clone().cpu()
        valid_sigmas = sorted([s for s, v in trajectory_map.items() if v is not None], reverse=True)
        valid_imgs = [trajectory_map[s] for s in valid_sigmas]
        fig, axes = plt.subplots(1, len(valid_sigmas), figsize=(15,2))
        if len(valid_sigmas) == 1:
            axes = [axes]
        for i, (sigma_val, img) in enumerate(zip(valid_sigmas, valid_imgs)):
            grid = torchvision.utils.make_grid(img, nrow=3, normalize=True, value_range=(-1,1))
            axes[i].imshow(grid.permute(1,2,0).cpu().numpy())
            axes[i].set_title(f"σ={sigma_val:.1f}")
            axes[i].axis('off')
        plt.suptitle(f"Denoising trajectory – class {cls_idx} | {SOLVER} | {TARGET}")
        plt.tight_layout()
        plt.show()

    print("Trajectory example (class 5):")
    sample_trajectory(cls_idx=5)

print("✅ Done. All cells executed.")